# 1.3 Условия, циклы, функции: решения упражнений

Аналитика 101 · блок «Дата аналитика» · тема 01 «Python для анализа» · страница 1.3.

Сначала идут восемь блоков кода со страницы: упражнения берут из них функцию `clean_brand`. Решения с разбором — во второй половине. Ноутбук выполнен целиком сверху вниз, все проверки прошли.

Загляните сюда после своей попытки: чужое решение понятнее, когда уже есть своё.

In [1]:
import platform

print("Python", platform.python_version())   # версия, на которой запущен ноутбук

Python 3.13.18


## Подготовка файлов

In [2]:
# Если учебных файлов нет рядом с ноутбуком, ячейка создаст их сама
import os

if not os.path.isdir("exports"):            # блоку про список файлов нужны только имена, поэтому файлы пустые
    os.makedirs("exports")
    for name in ["README.txt", "orders_2026-11-01.csv", "orders_2026-11-02.csv", "orders_2026-11-03.csv",
                 "orders_2026-11-03.xlsx", "returns_2026-11-02.csv", "~$orders_2026-11-03.xlsx"]:
        open(os.path.join("exports", name), "w").close()

if not os.path.exists("orders_big.csv"):    # то же, что делает make_orders_big.py
    pattern = [(" lumo", 1200, "paid"), ("BRISA", 850, "cancelled"), ("Lumo ", 2300, "paid"), ("kedr", 640, "paid"),
               ("brisa", 1750, "paid"), ("KEDR ", 990, "cancelled"), ("Lumo", 1430, "paid"), ("Brisa ", 560, "paid")]
    with open("orders_big.csv", "w", encoding="utf-8", newline="\n") as f:
        f.write("order_id,brand,amount,status\n")
        n = 0
        for _ in range(125_000):
            for brand, amount, status in pattern:
                n += 1
                f.write(f"{n},{brand},{amount},{status}\n")

print("файлов в exports:", len(os.listdir("exports")), "| orders_big.csv:", os.path.getsize("orders_big.csv"), "байт")

файлов в exports: 7 | orders_big.csv: 23513925 байт


## Код страницы

### Условие, for, enumerate, zip и while на суммах заказов

In [3]:
amounts = [1200, 850, 2300, 640, 1750]           # суммы заказов из страницы 1.1, ₽
brands = ["Lumo", "Brisa", "Lumo", "Kedr", "Brisa"]

# if / elif / else: срабатывает первая истинная ветка, остальные не проверяются
for a in amounts[:3]:
    if a >= 2000:
        print(a, "крупный")
    elif a >= 1000:
        print(a, "средний")
    else:
        print(a, "мелкий")

# enumerate: номер и элемент сразу; zip: два списка в ногу
for i, brand in enumerate(brands[:2], start=1):
    print(i, brand)
for brand, a in zip(brands[:2], amounts):
    print(brand, a)

# while: сколько первых заказов нужно, чтобы выручка дошла до 5000 ₽
total, n = 0, 0
while total < 5000 and n < len(amounts):   # второе условие — страховка от выхода за край
    total += amounts[n]
    n += 1
print(n, total)

1200 средний
850 мелкий
2300 крупный
1 Lumo
2 Brisa
Lumo 1200
Brisa 850
5 6740


### Рекурсия: сколько товаров в каталоге

In [4]:
import sys

# Учебный каталог магазина: в разделе либо подразделы, либо число товаров
catalog = {"Одежда": {"Куртки": 12, "Футболки": 30},
           "Обувь": {"Кеды": 8, "Ботинки": {"Зимние": 5, "Демисезонные": 7}}}


def count_items(node):
    if isinstance(node, int):         # базовый случай: дальше спускаться некуда
        return node
    total = 0
    for child in node.values():
        total += count_items(child)   # функция вызывает сама себя
    return total


print(count_items(catalog))
print(count_items(catalog["Обувь"]))
print(sys.getrecursionlimit())        # предел глубины стека по умолчанию

62
20
1000


Глубже всего стек на «Зимних» ботинках: поверх глобальной области четыре кадра count_items — для каталога, «Обуви», «Ботинок» и самого числа 5.

### Значение по умолчанию создаётся один раз

In [5]:
def f(x, acc=[]):        # список создаётся один раз — когда выполняется def
    acc.append(x)
    return acc

print(f(1))
print(f(2))              # тот же список, что и в первом вызове
print(f.__defaults__)    # он хранится в самой функции


def g(x, acc=None):      # исправление: None по умолчанию, новый список внутри
    if acc is None:
        acc = []
    acc.append(x)
    return acc

print(g(1))
print(g(2))

[1]
[1, 2]
([1, 2],)
[1]
[2]


### Переменная цикла после цикла

На странице этот блок заканчивается ошибкой `NameError`: переменной `late` нет, потому что цикл не сделал ни одной итерации. Здесь последняя строка обёрнута в `try`/`except`, чтобы ноутбук выполнялся до конца. Сама конструкция — тема страницы 1.4.

In [6]:
rows = [{"brand": " lumo", "amount": 1200}, {"brand": "BRISA", "amount": 850}]

for row in rows:
    pass
print(row)               # переменная цикла пережила цикл: последняя строка

amounts = [r["amount"] for r in rows]
print("r" in globals())  # переменная comprehension наружу не выходит

for late in []:          # пустая выгрузка: ни одной итерации
    pass
try:
    print(late)
except NameError as e:   # на странице здесь скрипт падает
    print("NameError:", e)

{'brand': 'BRISA', 'amount': 850}
False
NameError: name 'late' is not defined


### Вложенные if и ранний return дают один ответ

In [7]:
def order_label(row):            # вложенные if: логика уходит вправо
    if row["status"] == "paid":
        if row["amount"] is not None:
            if row["amount"] >= 2000:
                return "крупный"
            else:
                return "обычный"
        else:
            return "нет суммы"
    else:
        return "не оплачен"


def order_label_flat(row):       # ранний return: сначала отсекаем особые случаи
    if row["status"] != "paid":
        return "не оплачен"
    if row["amount"] is None:
        return "нет суммы"
    if row["amount"] >= 2000:
        return "крупный"
    return "обычный"


test = [{"status": "paid", "amount": 2300}, {"status": "paid", "amount": 640},
        {"status": "paid", "amount": None}, {"status": "cancelled", "amount": 850}]
print([order_label(r) for r in test])
print([order_label_flat(r) for r in test])

['крупный', 'обычный', 'нет суммы', 'не оплачен']
['крупный', 'обычный', 'нет суммы', 'не оплачен']


### Функция очистки бренда и выручка по брендам

In [8]:
# Учебная выгрузка заказов: бренд записан как попало
rows = [
    {"brand": " lumo",  "amount": 1200, "status": "paid"},
    {"brand": "BRISA",  "amount": 850,  "status": "cancelled"},
    {"brand": "Lumo ",  "amount": 2300, "status": "paid"},
    {"brand": "kedr",   "amount": 640,  "status": "paid"},
    {"brand": "brisa",  "amount": 1750, "status": "paid"},
    {"brand": "KEDR ",  "amount": 990,  "status": "cancelled"},
    {"brand": "Lumo",   "amount": 1430, "status": "paid"},
    {"brand": "Brisa ", "amount": 560,  "status": "paid"},
]

ALIASES = {"бриза": "Brisa", "кедр": "Kedr"}  # русские написания тех же брендов


def clean_brand(name, default="Без бренда"):
    """Приводит название бренда к одному написанию."""
    if not name:                           # None и пустая строка
        return default
    key = " ".join(name.split()).lower()   # лишние пробелы и регистр
    return ALIASES.get(key, key.title())


for raw in [" lumo", "KEDR ", "brisa  ", "Бриза", "", None]:
    print(f"{raw!r:>10} → {clean_brand(raw)}")

# Проверка, преобразование и агрегат — один цикл без копипаста
totals = {}
for row in rows:
    if row["status"] == "paid":
        brand = clean_brand(row["brand"])
        totals[brand] = totals.get(brand, 0) + row["amount"]
print(totals)

# Доли брендов в выручке — dict comprehension
total = sum(totals.values())
shares = {brand: round(s / total * 100) for brand, s in totals.items()}
print(total, shares)

   ' lumo' → Lumo
   'KEDR ' → Kedr
 'brisa  ' → Brisa
   'Бриза' → Brisa
        '' → Без бренда
      None → Без бренда
{'Lumo': 4930, 'Kedr': 640, 'Brisa': 2310}
7880 {'Lumo': 63, 'Kedr': 8, 'Brisa': 29}


`name.split()` без аргументов режет строку по любым пробелам и выбрасывает пустые куски, а `" ".join(…)` склеивает их обратно через один пробел. Так « Lumo  Kids » и «lumo kids» станут одним брендом.

### Comprehension для списка файлов

In [9]:
import os

files = sorted(os.listdir("exports"))
print(files)

# Только выгрузки заказов в CSV: без возвратов, файлов Excel и README
orders_files = [f for f in files
                if f.startswith("orders_") and f.endswith(".csv")]
print(orders_files)

# dict comprehension: дата из имени файла → имя файла
by_date = {f[7:17]: f for f in orders_files}
print(by_date)

['README.txt', 'orders_2026-11-01.csv', 'orders_2026-11-02.csv', 'orders_2026-11-03.csv', 'orders_2026-11-03.xlsx', 'returns_2026-11-02.csv', '~$orders_2026-11-03.xlsx']
['orders_2026-11-01.csv', 'orders_2026-11-02.csv', 'orders_2026-11-03.csv']
{'2026-11-01': 'orders_2026-11-01.csv', '2026-11-02': 'orders_2026-11-02.csv', '2026-11-03': 'orders_2026-11-03.csv'}


В учебной папке exports семь файлов, их список — первая строка вывода. Comprehension разбит на две строки по скобкам: так он читается как фраза «файлы, которые начинаются с orders_ и кончаются на .csv».

### Генератор для файла, который не влезает в память

In [10]:
import tracemalloc


def read_orders(path):
    """Отдаёт строки файла по одной: в памяти лежит одна строка, а не весь файл."""
    with open(path, encoding="utf-8") as f:
        header = next(f).rstrip("\n").split(",")
        for line in f:
            yield dict(zip(header, line.rstrip("\n").split(",")))


rows = read_orders("orders_big.csv")   # файл ещё не открыт: тело функции не началось
print(type(rows).__name__)
print(next(rows))                      # первая строка — по запросу

tracemalloc.start()
total = sum(int(r["amount"]) for r in read_orders("orders_big.csv")
            if r["status"] == "paid")
print(f"генератор: {total} ₽, пик памяти {tracemalloc.get_traced_memory()[1] / 2**20:.2f} МБ")
tracemalloc.stop()

tracemalloc.start()
all_rows = list(read_orders("orders_big.csv"))   # весь миллион строк в памяти
total = sum(int(r["amount"]) for r in all_rows if r["status"] == "paid")
print(f"список:    {total} ₽, пик памяти {tracemalloc.get_traced_memory()[1] / 2**20:.0f} МБ")
tracemalloc.stop()

generator
{'order_id': '1', 'brand': ' lumo', 'amount': '1200', 'status': 'paid'}


генератор: 985000000 ₽, пик памяти 0.03 МБ


список:    985000000 ₽, пик памяти 359 МБ


Учебный файл orders_big.csv — восемь строк выгрузки, повторённые 125 000 раз: миллион строк, около 23 МБ на диске. Пик памяти меряет tracemalloc из стандартной библиотеки, на другой машине числа будут немного другими. Конструкция with и чтение файлов подробно — на странице 1.4.

## Упражнения

Сначала заведём данные заново: после блока про генератор имя `rows` указывает уже не на список.

In [11]:
amounts = [1200, 850, 2300, 640, 1750]   # суммы заказов, ₽
rows = [                                 # та же выгрузка из восьми заказов, что в блоке про бренды
    {"brand": " lumo",  "amount": 1200, "status": "paid"},
    {"brand": "BRISA",  "amount": 850,  "status": "cancelled"},
    {"brand": "Lumo ",  "amount": 2300, "status": "paid"},
    {"brand": "kedr",   "amount": 640,  "status": "paid"},
    {"brand": "brisa",  "amount": 1750, "status": "paid"},
    {"brand": "KEDR ",  "amount": 990,  "status": "cancelled"},
    {"brand": "Lumo",   "amount": 1430, "status": "paid"},
    {"brand": "Brisa ", "amount": 560,  "status": "paid"},
]
print(len(amounts), "сумм,", len(rows), "строк выгрузки")

5 сумм, 8 строк выгрузки


### Упражнение 1. Копипаст → цикл

Цены со скидкой 10% посчитаны копипастом, и только для трёх заказов из пяти:

```python
price_1 = round(1200 * 0.9)
price_2 = round(850 * 0.9)
price_3 = round(2300 * 0.9)
```

Перепишите расчёт циклом по списку `amounts`. Результат — список `discounted` из пяти целых чисел.

In [12]:
discounted = []
for a in amounts:
    discounted.append(round(a * 0.9))
print(discounted)

[1080, 765, 2070, 576, 1575]


In [13]:
assert discounted == [1080, 765, 2070, 576, 1575], f"Ожидался список [1080, 765, 2070, 576, 1575], получено {discounted!r}"
print("Упражнение 1: верно")

Упражнение 1: верно


**Разбор.** Три строки отличались одним числом. Это признак цикла: число стало переменной `a`, а повтор взял на себя `for`. Заодно расчёт перестал зависеть от того, сколько заказов в списке.

### Упражнение 2. Условие → функция

Первый блок кода печатает метку заказа через `if`/`elif`/`else`. Сделайте из этого условия функцию `size_label(amount)`, которая метку возвращает: от 2000 ₽ — `"крупный"`, от 1000 ₽ — `"средний"`, меньше — `"мелкий"`. Если суммы нет (`None`), функция возвращает `"нет суммы"`. Используйте ранний `return`.

In [14]:
def size_label(amount):
    if amount is None:       # особый случай — первым, как в order_label_flat на странице
        return "нет суммы"
    if amount >= 2000:
        return "крупный"
    if amount >= 1000:
        return "средний"
    return "мелкий"


print([size_label(a) for a in amounts], size_label(None))

['средний', 'мелкий', 'крупный', 'мелкий', 'средний'] нет суммы


In [15]:
got = [size_label(a) for a in [2300, 2000, 1999, 1000, 999, None]]
assert got == ["крупный", "крупный", "средний", "средний", "мелкий", "нет суммы"], f"Получено {got!r}"
print("Упражнение 2: верно")

Упражнение 2: верно


**Разбор.** Проверка на `None` стоит первой: сравнение `None >= 2000` закончилось бы ошибкой `TypeError`. После каждого `return` функция уже вышла, поэтому `elif` и `else` не нужны, а код остаётся плоским.

### Упражнение 3. Цикл → comprehension

Цикл собирает суммы оплаченных заказов от 1000 ₽:

```python
paid_big = []
for row in rows:
    if row["status"] == "paid" and row["amount"] >= 1000:
        paid_big.append(row["amount"])
```

Запишите то же одним list comprehension.

In [16]:
paid_big = [row["amount"] for row in rows
            if row["status"] == "paid" and row["amount"] >= 1000]
print(paid_big)

[1200, 2300, 1750, 1430]


In [17]:
assert paid_big == [1200, 2300, 1750, 1430], f"Ожидался список [1200, 2300, 1750, 1430], получено {paid_big!r}"
print("Упражнение 3: верно")

Упражнение 3: верно


**Разбор.** Три части цикла переехали в одну запись: что кладём — слева, откуда берём — в середине, какие строки пропускаем — справа. Перенос строки внутри скобок разрешён, и условие на отдельной строке читается легче.

### Упражнение 4. Comprehension → цикл

Однострочник ниже собирает по каждому бренду список сумм оплаченных заказов. Считает он верно, но читать его трудно, и для каждой оплаченной строки он заново перебирает всю выгрузку.

Разверните его в обычный цикл с одним проходом по `rows`. Результат — словарь `by_brand`: бренд → список сумм.

In [18]:
dense = {clean_brand(r["brand"]): [x["amount"] for x in rows
                                   if x["status"] == "paid" and clean_brand(x["brand"]) == clean_brand(r["brand"])]
         for r in rows if r["status"] == "paid"}
print(dense)

{'Lumo': [1200, 2300, 1430], 'Kedr': [640], 'Brisa': [1750, 560]}


In [19]:
by_brand = {}
for row in rows:
    if row["status"] == "paid":
        brand = clean_brand(row["brand"])
        if brand not in by_brand:
            by_brand[brand] = []             # первый заказ бренда: заводим пустой список
        by_brand[brand].append(row["amount"])
print(by_brand)

{'Lumo': [1200, 2300, 1430], 'Kedr': [640], 'Brisa': [1750, 560]}


In [20]:
assert by_brand == {"Lumo": [1200, 2300, 1430], "Kedr": [640], "Brisa": [1750, 560]}, f"Получено {by_brand!r}"
print("Упражнение 4: верно")

Упражнение 4: верно


**Разбор.** Comprehension хорош, пока читается как одна фраза. Здесь внутри одного сидит второй, и на шесть оплаченных строк приходится шесть полных проходов по выгрузке. Цикл длиннее, зато проходит выгрузку один раз, а `clean_brand` вызывает один раз на строку.

### Упражнение 5. Список → генератор

Функция `paid_amounts` собирает суммы оплаченных заказов в список. Напишите генератор `iter_paid_amounts(rows)`, который отдаёт те же суммы по одной через `yield`.

In [21]:
def paid_amounts(rows):
    result = []
    for row in rows:
        if row["status"] == "paid":
            result.append(row["amount"])
    return result


print(paid_amounts(rows))

[1200, 2300, 640, 1750, 1430, 560]


In [22]:
def iter_paid_amounts(rows):
    for row in rows:
        if row["status"] == "paid":
            yield row["amount"]      # отдать сумму и замереть до следующего запроса


it = iter_paid_amounts(rows)
print(type(it).__name__, next(it), next(it), sum(iter_paid_amounts(rows)))

generator 1200 2300 7880


In [23]:
import inspect

g = iter_paid_amounts(rows)
assert inspect.isgenerator(g), "iter_paid_amounts должна быть генератором: в теле функции нужен yield"
assert next(g) == 1200, "Первое значение генератора — 1200"
assert sum(iter_paid_amounts(rows)) == 7880, "Сумма оплаченных заказов — 7880 ₽"
print("Упражнение 5: верно")

Упражнение 5: верно


**Разбор.** Список `result` и `return` исчезли: генератор ничего не накапливает. На восьми строках разницы нет, на миллионе она такая же, как в блоке про `orders_big.csv`: сотни мегабайт против сотых долей.